# Analyse de système de Balle sur poutre 

In [8]:
## importations ne pas modifier

from Models.BallBeam import ballbeam_config
from Models.BallBeam.StateSpace import LinearStateSpaceModel
from Models.BallBeam.TransferFunctions import TransferFunctionModel
from Models.BallBeam.NonlinearDynamics import NonlinearBallBeamModel

from Simulation.simulation import TFSimulator
from Simulation.simulation import HybridSim
from Simulation.simulation import NonLinearHybridSim

from Simulation.runners import *

from Metrics_Plotting.SimLog import SimLog
from Metrics_Plotting.Plotting import Plotting
from Metrics_Plotting.Metrics import Metrics

from Control.DiscretePID import DiscretePID
from Control.RSTController import RSTController

from Utils import computeRST
from Utils import utils

import numpy as np
import control as ct
import matplotlib.pyplot as plt

%matplotlib inline

Dans un notebook Jupyter, les cellules doivent généralement être exécutées dans l’ordre où elles apparaissent. En effet, certaines cellules définissent des variables, des fonctions ou des paramètres qui sont utilisés par les cellules suivantes. 

Si une cellule est exécutée avant celles dont elle dépend, des erreurs peuvent apparaître car certaines données ne seront pas encore disponibles en mémoire. Il est donc recommandé de commencer par la première cellule du notebook et de progresser séquentiellement jusqu’à la dernière. Lorsqu’une cellule est modifiée, il peut être nécessaire de réexécuter les cellules qui suivent afin de mettre à jour les résultats.

 Pour repartir d’un état propre, il est possible de redémarrer le noyau d’exécution (kernel) puis d’exécuter à nouveau toutes les cellules dans l’ordre. Cette pratique permet de vérifier que le notebook fonctionne correctement et que tous les résultats peuvent être reproduits à partir d’une session vierge.

Pour éxécuter toutes les cellules dans l'ordre descendent cliquer sur Run->Run all cells

 Pour redémarer le noyeau cliquer sur Kernel -> Restart Kernel


### Configuration de la simulation:


T est le temps total de simulation et dt est le temps d'échantillonage du correcteur

In [9]:
ballbeam_config.T=6  #sec
ballbeam_config.dt=0.05 #secv

##### Initalisation des simulateurs
Dans cette partie, on crée les instances de classes nécesaaires à la simulation de la réponse indicielle et impulsionelle

In [10]:
X_0 = np.array([[0.0],[0]])                                             # état initial
double_int = TransferFunctionModel(ballbeam_config)                     # instance de TransferFunctionModel
double_int_NL =NonlinearBallBeamModel(ballbeam_config)                  # pour la simulation hybride non linéaire
double_int_simNL=NonLinearHybridSim(double_int_NL,ballbeam_config)      # simulation continue
Double_int_sim = TFSimulator(double_int.Tf_dis, X_0)                    # simulateur de fonctions de transfert discrètes (double_int.Tf_Dis)
A = double_int.Tf_dis.den_list[0][0]                                    # extraction du dénominateur A(z)
B = double_int.Tf_dis.num_list[0][0]                                    # extraction du numérateur B(z)

In [11]:
### affichage 
print(double_int.Tf_cont)  ## la fonction de transfert continue
print(double_int.Tf_dis) ## la fonction de transfert discrète 

<TransferFunction>: sys[4]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']

  0.21
  ----
  s^2
<TransferFunction>: sys[4]$sampled
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = 0.05

  0.0002625 z + 0.0002625
  -----------------------
       z^2 - 2 z + 1


### Réponse Impulsionelle